# CQ Pipeline Walkthrough

Stage 9 tutorial: generate a CQ trace, simulate it, and visualise the dispatcher timeline.

**Prerequisites**
- Run this notebook from the project root (`IA_RISC_V_NPU_Simulator`).
- Install dependencies: `pip install -r ia_risc_v_npu/requirements.txt`.
- Optional: `pip install plotly` for interactive visualisation.

In [ ]:
from pathlib import Path
import tempfile

REPO_ROOT = Path.cwd().resolve()
PKG_ROOT = REPO_ROOT / "ia_risc_v_npu"
TRACE_SPEC = REPO_ROOT / "workloads" / "cq" / "sample_gemm.yaml"
artifacts_dir = Path(tempfile.mkdtemp(prefix="cq_pipeline_"))
TRACE_JSONL = artifacts_dir / "sample_gemm.jsonl"
CQ_SUMMARY = artifacts_dir / "cq_simulated.json"

print(f"Artifacts will be written to: {artifacts_dir}")

In [ ]:
import subprocess

subprocess.run(
    [
        "python",
        "-m",
        "src.cq.tools.plan_generator",
        "--input",
        str(TRACE_SPEC),
        "--output",
        str(TRACE_JSONL),
    ],
    check=True,
    cwd=PKG_ROOT,
)
subprocess.run(
    [
        "python",
        "-m",
        "src.simulator.cli",
        "run-cq",
        str(TRACE_JSONL),
        "--simulate",
        "--output",
        str(CQ_SUMMARY),
    ],
    check=True,
    cwd=PKG_ROOT,
)
print("CQ simulation summary ready:", CQ_SUMMARY)

In [ ]:
import json

payload = json.loads(CQ_SUMMARY.read_text(encoding="utf-8"))
cq_exec = payload.get("cq_execution", payload)
timeline = cq_exec["dispatch"]["timeline"]
timeline

In [ ]:
try:
    import plotly.express as px
except ImportError:
    print("Plotly not installed; run `pip install plotly` to enable visualisation.")
else:
    fig = px.timeline(
        timeline,
        x_start="start_tick",
        x_end="end_tick",
        y="lane",
        color="lane",
        hover_data=["cmd_id", "duration_ticks"],
    )
    fig.update_layout(title="CQ Dispatcher Timeline")
    fig.show()

## Next Steps
- Use `python -m src.scripts.cq_timeline_export` to persist CSV timelines for reports.
- Swap `workloads/cq/sample_gemm.yaml` with a golden workload to compare dispatcher policies.
- Regenerate `docs/reference/isa_cq_reference.md` after ISA/CQ spec edits.